# Chapter 14 — The Model Is Not the Process

**Companion to Applied AI**

Question: Can the process restart honestly after a crash — without redoing effects?

By the end of this notebook you will have:

- separated task, state, model call, verification, and effect
- restarted from persisted task state instead of replaying the model
- shown completion derived from acceptance, not from generation

## What this notebook demonstrates
A five-stage process where each stage is a separate, persisted event. A simulated crash between verification and effect shows why `generation ≠ completed`.

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)
import json

seed: 42


## 1. The stages, as data

In [2]:
store = {"events": []}
def record(kind: str, data: dict):
    store["events"].append({"kind": kind, **data})

record("task", {"task_id": "t-1", "goal": "write deploy note"})
record("model_call", {"task_id": "t-1", "proposal": "canary 10%"})
record("verification", {"task_id": "t-1", "check": "budget<=limit", "passed": True})
print([e["kind"] for e in store["events"]])

['task', 'model_call', 'verification']


## 2. Crash before the effect — then restart from state, not from the model

In [3]:
# crash: process dies here; `store` survives (imagine it on disk)
snapshot = json.loads(json.dumps(store))  # persisted state

def current_status(events, task_id):
    kinds = [e["kind"] for e in events if e["task_id"] == task_id]
    if "acceptance" in kinds: return "completed"
    if "effect" in kinds: return "effected-unaccepted"
    if "verification" in kinds: return "verified-uneffected"
    return "proposed"

print("status after crash:", current_status(snapshot["events"], "t-1"))
assert current_status(snapshot["events"], "t-1") == "verified-uneffected"

status after crash: verified-uneffected


## 3. Resume: perform the effect once, then accept it

In [4]:
record("effect", {"task_id": "t-1", "action": "note written"})
record("acceptance", {"task_id": "t-1", "accepted_by": "verifier"})
print("status after resume:", current_status(store["events"], "t-1"))
assert current_status(store["events"], "t-1") == "completed"
assert sum(1 for e in store["events"] if e["kind"] == "effect") == 1, "effect happened exactly once"

status after resume: completed


## Interpretation
- Supports: completion is derived from an acceptance event caused by a verified effect — never from the model call alone.
- Does NOT support: a production crash-recovery protocol (no concurrency here).

## Try it yourself
1. Crash *after* the effect but before acceptance; show resume must check before re-acting.
2. Edit the proposal after verification and require re-verification.
3. Hand-write an `acceptance` event with no effect and write a checker that refuses it.